# 11: O Consolidador (relatório final)

Toda análise termina no Consolidador. Antes dele, a resposta final era só a última mensagem do grafo:
às vezes o texto do setor, às vezes o diagnóstico, nunca tudo junto.

O Consolidador **não é um agente com tools**: é uma única chamada ao LLM com saída estruturada
(`RelatorioFinal`), seguida de ajustes em código e da formatação em markdown.

In [1]:
import sys
import warnings
from pathlib import Path

# Sobe pastas até achar a raiz do projeto (onde está o pyproject.toml), de onde quer que o notebook seja aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ))

from dotenv import load_dotenv
_ = load_dotenv(RAIZ / ".env")

# Aviso de depreciação interno do LangGraph, sem relação com o projeto: só polui a saída.
# O filtro vem depois do import do langchain_core, que instala o filtro dele ao ser importado.
from langchain_core._api.deprecation import LangChainPendingDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainPendingDeprecationWarning)
warnings.filterwarnings("ignore", message="Function wrap_gemini is in beta")

## 1. Uma análise completa da VALE3

Esta célula roda o pipeline inteiro (cerca de 30 segundos). A VALE3 costuma acionar o Investigator, então o
relatório terá diagnóstico.

In [2]:
from langchain_core.messages import HumanMessage

from fluxo_financeiro import app

resultado = app.invoke({"messages": [HumanMessage("Analise a ação VALE3")]})
quem_falou = [m.name for m in resultado["messages"] if m.name]
print("Mensagens no histórico:", quem_falou)
print("loop_count:", resultado["loop_count"])

2026-11-08 13:20:28,242 - fluxo_financeiro - INFO - [supervisor] inicio — loop_count=0


2026-11-08 13:20:29,680 - fluxo_financeiro - INFO - [supervisor] decisao=YFinance | tarefa='Colete os dados financeiros, histórico de cotação e principais métricas fundamentalistas para a ação VALE3.'


2026-11-08 13:20:29,681 - fluxo_financeiro - DEBUG - [router] YFinance → YFinance


2026-11-08 13:20:29,683 - fluxo_financeiro - INFO - [node] YFinance: invocando agente (2 msgs)


2026-11-08 13:20:30,454 - fluxo_financeiro - INFO - [tool] buscar_dados_acao_b3: ticker=VALE3.SA


2026-11-08 13:20:32,535 - fluxo_financeiro - INFO - [tool] buscar_dados_acao_b3: ok (2447 chars)


... (32 outputs left out)

2026-11-08 13:20:58,414 - fluxo_financeiro - DEBUG - [router] FINISH → Consolidador


2026-11-08 13:20:58,416 - fluxo_financeiro - INFO - [node] Consolidador: consolidando (9 msgs)


2026-11-08 13:21:00,885 - fluxo_financeiro - INFO - [node] Consolidador: relatório pronto (2936 chars)


Mensagens no histórico: ['Supervisor', 'YFinance', 'Supervisor', 'Analyzer', 'Supervisor', 'Analyzer', 'Supervisor', 'Investigator', 'Consolidador']
loop_count: 5


## 2. O que o Consolidador recebe e devolve

Ele recebe só o pedido do usuário e as respostas de YFinance, Analyzer e Investigator (as tarefas do
Supervisor ficam de fora), cada uma dentro de `<resposta_agente>`. Devolve um `RelatorioFinal`:

In [3]:
from fluxo_financeiro.agents.consolidator import montar_insumos

print("\n\n".join(montar_insumos(resultado["messages"]))[:1500], "...")

<pedido_usuario>Analise a ação VALE3</pedido_usuario>

<resposta_agente nome="YFinance" n="1">
Dados para VALE3.SA:
Empresa: Vale S.A.
Preço Atual: R$ 64.04 (no dia: 2.19%)
Faixa de 52 semanas: R$ 59.42 a R$ 91.62
Valor de mercado: R$ 248.5 bi
LPA: R$ 0.52
P/L:  77.2x
ROE:  0.3%
DY:   6.82%
Setor: Materiais Básicos | Subsetor: Mineração de Minerais Metálicos
Maiores do setor na Brapi (Non-Energy Minerals, por valor de mercado): VALE3 é a 1ª de 12.
  1. VALE3 VALE S.A.: R$ 248.5 bi
  2. GGBR4 GERDAU S.A.: R$ 46.0 bi
  3. CMIN3 CSN MINERAÇÃO S.A.: R$ 28.3 bi
  4. KLBN4 KLABIN S.A.: R$ 21.7 bi
  5. GOAU4 METALURGICA GERDAU S.A.: R$ 14.0 bi
Sumario de Negócio:
A Vale S.A. (VALE3) é uma companhia brasileira de mineração e metais, com foco principal na produção de minério de ferro e de metais para transição energética. A empresa também atua na produção de pelotas de minério de ferro e, no segmento de metais básicos, em produtos como níquel, cobre, cobalto e metais do grupo da platina, além d

In [4]:
relatorio = resultado["relatorio"]
for campo in ("resumo", "indicadores", "contexto", "riscos", "limitacoes"):
    print(f"--- {campo}:\n{getattr(relatorio, campo)}\n")

--- resumo:
A Vale (VALE3) é a maior mineradora da B3, cotada a R$ 64,04 com um P/L de 77,2x e Dividend Yield de 6,82%. O papel foi penalizado recentemente por uma queda de 53% no lucro líquido do trimestre (US$ 1,25 bi contra US$ 2,69 bi), reflexo da retração nos preços internacionais do minério de ferro e da menor demanda na China.

--- indicadores:
['Preço Atual: R$ 64.04', 'Faixa de 52 semanas: R$ 59.42 a R$ 91.62', 'Valor de mercado: R$ 248.5 bi', 'LPA: R$ 0.52', 'P/L: 77.2x', 'ROE: 0.3%', 'DY: 6.82%', 'Sharpe Ratio: -0.11', 'Sortino Ratio: -0.18', 'Max Drawdown: -31.5%', 'VaR 95% (diário): -2.91%', 'Beta (vs IBOV): 0.39', 'Alpha (anual): -3.4%', 'Piotroski F-Score: 5/9']

--- contexto:
A Vale atua nos segmentos de Soluções em Minério de Ferro e Metais para Transição Energética, operando sistemas integrados de mina, ferrovia e porto. O setor de mineração em 2026 enfrenta um momento complexo devido à retração nos preços internacionais do minério de ferro, estoques elevados e demand

## 3. O diagnóstico é copiado, não resumido

O LLM é instruído a deixar `diagnostico` vazio; o código copia o texto do Investigator literalmente. Assim as
quatro seções (Causa, Natureza, Evidências, Impacto) nunca são reescritas.

In [5]:
texto_investigator = next((m.text for m in resultado["messages"] if m.name == "Investigator"), None)
print("Investigator rodou:", texto_investigator is not None)
print("Diagnóstico idêntico ao texto do Investigator:", relatorio.diagnostico == texto_investigator)

Investigator rodou: True
Diagnóstico idêntico ao texto do Investigator: True


## 4. A mesma função, chamada à parte

`consolidar(messages)` é uma função comum: dá para chamá-la com qualquer histórico. Uma chamada ao LLM.

In [6]:
from fluxo_financeiro.agents.consolidator import consolidar, formatar_relatorio, relatorio_de_emergencia

de_novo = consolidar(resultado["messages"])
print("Indicadores na 2a chamada:", len(de_novo.indicadores), "| na análise:", len(relatorio.indicadores))

Indicadores na 2a chamada: 14 | na análise: 14


## 5. Do relatório estruturado ao markdown

`formatar_relatorio` monta o markdown em código. Seções vazias são omitidas.

In [7]:
from IPython.display import Markdown

Markdown(formatar_relatorio(relatorio, "VALE3"))

# Relatório: VALE3

## Resumo
A Vale (VALE3) é a maior mineradora da B3, cotada a R$ 64,04 com um P/L de 77,2x e Dividend Yield de 6,82%. O papel foi penalizado recentemente por uma queda de 53% no lucro líquido do trimestre (US$ 1,25 bi contra US$ 2,69 bi), reflexo da retração nos preços internacionais do minério de ferro e da menor demanda na China.

## Indicadores
- Preço Atual: R$ 64.04
- Faixa de 52 semanas: R$ 59.42 a R$ 91.62
- Valor de mercado: R$ 248.5 bi
- LPA: R$ 0.52
- P/L: 77.2x
- ROE: 0.3%
- DY: 6.82%
- Sharpe Ratio: -0.11
- Sortino Ratio: -0.18
- Max Drawdown: -31.5%
- VaR 95% (diário): -2.91%
- Beta (vs IBOV): 0.39
- Alpha (anual): -3.4%
- Piotroski F-Score: 5/9

## Notícias e setor
A Vale atua nos segmentos de Soluções em Minério de Ferro e Metais para Transição Energética, operando sistemas integrados de mina, ferrovia e porto. O setor de mineração em 2026 enfrenta um momento complexo devido à retração nos preços internacionais do minério de ferro, estoques elevados e demanda enfraquecida na China, embora a empresa mantenha seu guidance de produção para 2026 entre 335 e 345 milhões de toneladas e expanda a atuação em metais básicos.

## Diagnóstico da anomalia
- Causa identificada: A queda de 53% no lucro líquido do 3T26 em relação ao 3T26 (queda de US$ 2,69 bi para US$ 1,25 bi) decorreu fundamentalmente da retração nos preços internacionais do minério de ferro (pressionados pela demanda fraca na China) somada à compressão das margens por elevação de custos de produção e frete, além de efeitos cambiais. Não houve eventos extraordinários (como grandes impairments ou provisões one-off) reportados no trimestre.
- Natureza do evento: Cíclico (atrelado aos ciclos de commodities e preços internacionais do minério de ferro / demanda chinesa), com pressões conjunturais de custos operacionais.
- Evidências: Fatos relevantes e demonstrativos financeiros da Vale S.A., histórico trimestral de Lucro Líquido (US$ 1,25 bi no 3T26 vs US$ 2,69 bi no 3T25) e relatórios de mercado.
- Impacto no investimento: O resultado reforça a forte correlação da empresa com o ciclo do aço na China. Embora o negócio de minério continue gerando caixa de forma expressiva, a alta ciclicidade exige monitoramento da demanda asiática e da eficiência de custos, ao mesmo tempo em que a expansão em metais básicos (cobre e níquel) atua como avenida de diversificação e mitigação de risco de longo prazo.

## Riscos
- Retração nos preços internacionais do minério de ferro
- Compressão de margens por elevação de custos de produção e frete, além de efeitos cambiais
- Forte correlação com o ciclo do aço na China e demanda fraca na região
- Lucro líquido trimestral menor (queda de 53% no comparativo anual)
- Retração de 20% no Ebitda proforma
- Recuo de 6% na produção trimestral (88,4 milhões de toneladas)
- Dívida corrente aumentada (+21.2%)
- Gross Margin em queda (35.0% vs 36.2%)
- Alpha anual negativo de -3.4%


## 6. Se o LLM do Consolidador falhar

A análise já custou várias chamadas; perder tudo no último passo seria ruim. Se `consolidar` levantar erro
(cota, rede, saída inválida), o nó usa `relatorio_de_emergencia`: as respostas dos agentes juntadas, sem LLM,
e `relatorio = None` no estado.

In [8]:
print(relatorio_de_emergencia(resultado["messages"])[:800], "...")

> Não foi possível consolidar o relatório; seguem as respostas dos agentes.

## YFinance
Dados para VALE3.SA:
Empresa: Vale S.A.
Preço Atual: R$ 64.04 (no dia: 2.19%)
Faixa de 52 semanas: R$ 59.42 a R$ 91.62
Valor de mercado: R$ 248.5 bi
LPA: R$ 0.52
P/L:  77.2x
ROE:  0.3%
DY:   6.82%
Setor: Materiais Básicos | Subsetor: Mineração de Minerais Metálicos
Maiores do setor na Brapi (Non-Energy Minerals, por valor de mercado): VALE3 é a 1ª de 12.
  1. VALE3 VALE S.A.: R$ 248.5 bi
  2. GGBR4 GERDAU S.A.: R$ 46.0 bi
  3. CMIN3 CSN MINERAÇÃO S.A.: R$ 28.3 bi
  4. KLBN4 KLABIN S.A.: R$ 21.7 bi
  5. GOAU4 METALURGICA GERDAU S.A.: R$ 14.0 bi
Sumario de Negócio:
A Vale S.A. (VALE3) é uma companhia brasileira de mineração e metais, com foco principal na produção de minério de ferro e de metais para tra ...


## Sem recomendação de compra ou venda

O relatório descreve o que os dados mostram e não dá veredito nem preço-alvo. Um "compre" escrito por um LLM a
partir de poucas fontes passaria uma confiança que os dados não sustentam.

## O que observar

- `limitacoes`: lista o que faltou ou falhou (busca sem resultado, etapa que não rodou). Parte delas é escrita em código, não depende de o LLM perceber.
- Os números de `indicadores` precisam aparecer nas respostas dos agentes: o prompt proíbe inventar ou recalcular.
- A segunda chamada a `consolidar` pode escrever frases diferentes da primeira (o LLM não é determinístico), mas os números são os mesmos, porque vêm dos insumos.
- O relatório de emergência é feio, mas não perde nada do que os agentes produziram.